In [1]:
# grab directory root
import sys
sys.path.append("../")

In [2]:
from dinov3.eval.tSNE import extract_embeddings
from dinov3.data.datasets import NCells
from dinov3.models.backbone_loader import load_backbone
from dinov3.eval.simpleKNN import evaluate_simple_knn
import torch
import json
from torchvision import transforms


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/site-packages/torch/distributed/distributed_c10d.py:4631: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
  warnings.warn(  # warn only once
[rank0]:[W916 17:55:49.795463891 ProcessGroupNCCL.cpp:4718] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 as device used by this process is currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorrect. You can pecify device_id in init_process_group() to force use of a particular device.


In [3]:
# defaul transform used in dinov3
def make_transform(resize_size: int | list[int] = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

test_dataset = NCells(
    root="/home/students/code/helmholtzSS25/Hannes/dinov3playground/manifest_test_fixed.csv.gz",
    split=NCells.Split.TRAIN,
    transform=make_transform(), 
    target_transform=None,
)
print(f"Test Dataset contains {len(test_dataset)} entries")

Test Dataset contains 359426 entries


## Baseline kNN Evaluation

In [4]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/ckpt/123749",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 123750 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        

batches: 100%|██████████| 5617/5617 [05:56<00:00, 15.75it/s]


In [5]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.47364408807376207,
        "recall@k": 0.00022098644261653267,
        "topk_acc": 0.47364408807376207
    },
    "5": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.5153077406754102,
        "recall@k": 0.0009799491545323365,
        "topk_acc": 0.7874833762721672
    },
    "10": {
        "mAP": 0.5931701375128093,
        "precision@k": 0.49975627806558226,
        "recall@k": 0.0018330538328638808,
        "topk_acc": 0.8676166999604925
    }
}


In [10]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80/ckpt/123749",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-e80"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 123750 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        

batches:   0%|          | 19/5617 [00:04<19:46,  4.72it/s] 


KeyboardInterrupt: 

In [11]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.522654338451847,
        "precision@k": 0.4745816941456656,
        "recall@k": 0.0002209196774884609,
        "topk_acc": 0.4745816941456656
    },
    "5": {
        "mAP": 0.522654338451847,
        "precision@k": 0.45836862107916504,
        "recall@k": 0.0009804500411107458,
        "topk_acc": 0.6517335974581694
    },
    "10": {
        "mAP": 0.522654338451847,
        "precision@k": 0.45024455659857654,
        "recall@k": 0.001833994614048822,
        "topk_acc": 0.7318891788574005
    }
}


## DBScan (Single Cluster) Finetune Eval

In [4]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [09:46<00:00,  9.58it/s]


In [5]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5959169510237845,
        "precision@k": 0.5472559024667109,
        "recall@k": 0.000223021184014311,
        "topk_acc": 0.5472559024667109
    },
    "5": {
        "mAP": 0.5959169510237845,
        "precision@k": 0.5332691569335551,
        "recall@k": 0.0010033149822359709,
        "topk_acc": 0.723314395730971
    },
    "10": {
        "mAP": 0.5959169510237845,
        "precision@k": 0.5245956052149815,
        "recall@k": 0.0018819778263280305,
        "topk_acc": 0.8022958828799253
    }
}


## HDBScan Finetune Eval

In [6]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [06:19<00:00, 14.78it/s]


In [ ]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5820977197181092,
        "precision@k": 0.5480738733424961,
        "recall@k": 0.0002242270425645246,
        "topk_acc": 0.5480738733424961
    },
    "5": {
        "mAP": 0.5820977197181092,
        "precision@k": 0.47641906818093294,
        "recall@k": 0.0010104705236814104,
        "topk_acc": 0.7893947571961961
    },
    "10": {
        "mAP": 0.5820977197181092,
        "precision@k": 0.4962565312470439,
        "recall@k": 0.0018909567039771136,
        "topk_acc": 0.8677891972200119
    }
}


## HDBScan Eval (with min_smaple=2)

In [5]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan2/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan2/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan2"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches:   1%|          | 31/5617 [00:03<10:43,  8.68it/s] 


KeyboardInterrupt: 

In [5]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5937017718030762,
        "precision@k": 0.5456199607151403,
        "recall@k": 0.00022136790386864037,
        "topk_acc": 0.5456199607151403
    },
    "5": {
        "mAP": 0.5937017718030762,
        "precision@k": 0.5307351165469387,
        "recall@k": 0.0009926657076118983,
        "topk_acc": 0.719992432378292
    },
    "10": {
        "mAP": 0.5937017718030762,
        "precision@k": 0.5220740847907497,
        "recall@k": 0.001857161567762058,
        "topk_acc": 0.8003483331756746
    }
}


## HDBSCAN Negative Lambda 

In [6]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-hdbscan2/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambdas/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambdas"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [06:47<00:00, 13.80it/s]


In [7]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.6111058866861816,
        "precision@k": 0.47524664325897403,
        "recall@k": 0.00022349294359044075,
        "topk_acc": 0.47524664325897403
    },
    "5": {
        "mAP": 0.6111058866861816,
        "precision@k": 0.5029474773666902,
        "recall@k": 0.0010003710013825554,
        "topk_acc": 0.7867794761647738
    },
    "10": {
        "mAP": 0.6111058866861816,
        "precision@k": 0.5087113898271133,
        "recall@k": 0.0018657427246612367,
        "topk_acc": 0.8665650231201972
    }
}


HDBSCAN Lambda = 10

In [5]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/dinov3/configs/train/dinov3_neglambds.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambdas-10/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambdas-10/"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [06:46<00:00, 13.81it/s]


In [6]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5228901792717398,
        "precision@k": 0.4736357414321724,
        "recall@k": 0.00022006970281041431,
        "topk_acc": 0.4736357414321724
    },
    "5": {
        "mAP": 0.5228901792717398,
        "precision@k": 0.4597853243783144,
        "recall@k": 0.0009827637761918202,
        "topk_acc": 0.6500559224986506
    },
    "10": {
        "mAP": 0.5228901792717398,
        "precision@k": 0.45140529622230996,
        "recall@k": 0.0018408619507018441,
        "topk_acc": 0.7294157907330021
    }
}


In [13]:
# Lambda = 2 contrastive (SUM)

In [5]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive/"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches:   2%|▏         | 112/5617 [00:12<10:09,  9.03it/s]
Exception in thread Thread-6 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/students/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/students/.conda/envs/helmholtz_hannes/lib/python3.10/multipro

KeyboardInterrupt: 

In [9]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.581907148061912,
        "precision@k": 0.5362911976317796,
        "recall@k": 0.00022503564305786962,
        "topk_acc": 0.5362911976317796
    },
    "5": {
        "mAP": 0.581907148061912,
        "precision@k": 0.49176297763656496,
        "recall@k": 0.0009824975675120987,
        "topk_acc": 0.7744542687507303
    },
    "10": {
        "mAP": 0.581907148061912,
        "precision@k": 0.4815569825221325,
        "recall@k": 0.0018170308351911602,
        "topk_acc": 0.8568133635296278
    }
}


# Contrastive mean fucked margin, min_sample = 2

In [7]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean/"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [07:14<00:00, 12.94it/s]


In [8]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.599379580432497,
        "precision@k": 0.5513123702792786,
        "recall@k": 0.0002373887504300616,
        "topk_acc": 0.5513123702792786
    },
    "5": {
        "mAP": 0.599379580432497,
        "precision@k": 0.5361604335802085,
        "recall@k": 0.0010551516155077345,
        "topk_acc": 0.7269924824581416
    },
    "10": {
        "mAP": 0.599379580432497,
        "precision@k": 0.527094033264149,
        "recall@k": 0.0019603212546689657,
        "topk_acc": 0.8053618825571884
    }
}


# Contrastive mean fixed margin, min_sample = 2

In [9]:
model = load_backbone(
    config_file="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean_fixed_margin/config.yaml",
    pretrained_weights="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean_fixed_margin/ckpt/22499",
    output_dir="/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-neglambda2_contrastive_mean_fixed_margin/"
)
target_size = 224
batch_size = 64
num_workers = 6
embeddings, labels = extract_embeddings(
    model=model,
    dataset=test_dataset,
    device=DEVICE,
    batch_size=batch_size,
    num_workers=num_workers,
    target_size=target_size,
)

Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches: 100%|██████████| 5617/5617 [06:59<00:00, 13.41it/s]


In [10]:
results = evaluate_simple_knn(embeddings, labels, k_list=[1,5,10], metric='cosine', sample_size=None)
print("kNN Performance:", json.dumps(results, sort_keys=True, indent=4))

kNN Performance: {
    "1": {
        "mAP": 0.5868439601931816,
        "precision@k": 0.4766850478262563,
        "recall@k": 0.00022310338989768976,
        "topk_acc": 0.4766850478262563
    },
    "5": {
        "mAP": 0.5868439601931816,
        "precision@k": 0.4909138459655117,
        "recall@k": 0.0010030568460519475,
        "topk_acc": 0.7890664559603368
    },
    "10": {
        "mAP": 0.5868439601931816,
        "precision@k": 0.4684727871662039,
        "recall@k": 0.0018839005200982702,
        "topk_acc": 0.8678754458497716
    }
}


In [12]:
from collections import Counter

counts = Counter(labels)
total = len(labels)
balance = {lab: {"count": c, "pct": c/total} for lab, c in counts.items()}

# pretty print
for lab, stats in sorted(balance.items(), key=lambda x: (-x[1]["count"], x[0])):
    print(f"{lab}: {stats['count']} ({stats['pct']:.2%})")

N_tissuenet: 68617 (19.09%)
N_DynamicNuclearNet: 30904 (8.60%)
N_BCCD: 11839 (3.29%)
Control__B: 7500 (2.09%)
Control__DC: 7500 (2.09%)
Control__M0: 7500 (2.09%)
Control__Negs: 7500 (2.09%)
Control__Nk: 7500 (2.09%)
Control__T0: 7500 (2.09%)
Control__T4: 7500 (2.09%)
Control__T8: 7500 (2.09%)
Multiplexed__B: 7500 (2.09%)
Multiplexed__DC: 7500 (2.09%)
Multiplexed__M0: 7500 (2.09%)
Multiplexed__Negs: 7500 (2.09%)
Multiplexed__Nk: 7500 (2.09%)
Multiplexed__T0: 7500 (2.09%)
Multiplexed__T4: 7500 (2.09%)
Multiplexed__T8: 7500 (2.09%)
Epithelial: 6391 (1.78%)
N_Neurips: 5974 (1.66%)
Breast__Neoplastic: 5493 (1.53%)
N_cyto2: 4900 (1.36%)
kumar: 4483 (1.25%)
bact_fluor_wiggins: 4397 (1.22%)
Lymphocyte: 4170 (1.16%)
Breast__Epithelial: 3931 (1.09%)
Uterus__Neoplastic: 3714 (1.03%)
negative: 3424 (0.95%)
Breast__Connective: 3077 (0.86%)
positive: 2999 (0.83%)
cpm17: 2705 (0.75%)
bact_phase_wiggins: 2646 (0.74%)
Colon__Connective: 2545 (0.71%)
N_yeaz: 2192 (0.61%)
bact_phase_PSVB: 2184 (0.61%)
Co

Contrastive 